# Cats vs dogs with binary Logistic Regression

## 1. Objective and concepts
Cat is 0 and Dog is 1. The model computes a linear score $z=w^Tx+b$, then the sigmoid $p(\mathrm{Dog}|x)=1/(1+e^{-z})$. We predict Dog when its estimated probability is at least 0.5, otherwise Cat. The number is a model estimate and is not a calibrated guarantee.

## 2. Libraries and data
NumPy represents normalized pixels, pandas reads the saved split manifest, and Matplotlib shows predictions. scikit-learn provides Logistic Regression, fold-specific scaling and evaluation. SciPy checks the probability formula, and joblib saves the fitted pipeline. We use the fixed manifest from notebook 01.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'common.py').exists() else Path.cwd().parent
assert (ROOT / 'common.py').exists(), 'Open this notebook from the project or notebooks folder.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from common import LABELS, SEED, pixels
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy.special import expit, softmax
from threadpoolctl import threadpool_limits
import joblib
from common import evaluate, show_predictions
manifest=pd.read_csv(ROOT / 'data/processed/split_manifest.csv')
arrays=np.load(ROOT / 'data/processed/images.npz')
assert np.array_equal(manifest.image_id.to_numpy(), arrays['image_ids'])
X=arrays['pixels24'].reshape(len(manifest),-1).astype(np.float32)/255.0


keep=manifest.label.isin(['Cat','Dog']).to_numpy()
manifest=manifest.loc[keep].reset_index(drop=True); X=X[keep]
labels=LABELS[:2]
y=manifest.label.map({'Cat':0,'Dog':1}).to_numpy()

train=manifest.split.eq('train').to_numpy()
validation=manifest.split.eq('validation').to_numpy()
test=manifest.split.eq('test').to_numpy()
display(pd.crosstab(manifest.label,manifest.split))
print('Features per image:',X.shape[1])

## 3. Cross-validation and regularization
Three stratified group folds within the training split select C from 0.0001, 0.001 and 0.01 using macro-F1. Each fold trains on two parts and scores the remaining part. Related-image groups stay intact. StandardScaler is inside the pipeline, so the held-out fold cannot influence its mean or scale.

C controls inverse regularization strength: smaller C penalizes large coefficients more strongly. Mean fold scores compare these choices; their variation describes sensitivity to the fold partition, not a confidence interval. The separate validation and final test images do not select C.

In [ ]:
pipeline=Pipeline([('scale',StandardScaler()),
                   ('classifier',LogisticRegression(solver='lbfgs',max_iter=600,tol=1e-4,random_state=SEED))])
cv=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)
# Verify group isolation in every cross-validation fold.
for fit_idx,score_idx in cv.split(X[train],y[train],manifest.loc[train,'group_id']):
    groups=manifest.loc[train,'group_id'].to_numpy()
    assert set(groups[fit_idx]).isdisjoint(groups[score_idx])
search=GridSearchCV(pipeline,{'classifier__C':[0.0001,0.001,0.01]},scoring='f1_macro',
                    cv=cv,n_jobs=1,return_train_score=True,error_score='raise')
started=time.perf_counter()
with threadpool_limits(limits=4):
    search.fit(X[train],y[train],groups=manifest.loc[train,'group_id'])
elapsed=time.perf_counter()-started
model=search.best_estimator_
cv_results=pd.DataFrame(search.cv_results_)
cv_results.to_csv(ROOT / 'outputs/binary_cross_validation.csv',index=False)
display(cv_results[['param_classifier__C','mean_train_score','mean_test_score','std_test_score']])
print('Selected C:',search.best_params_['classifier__C'],'Training and CV seconds:',round(elapsed,2))
assert model['classifier'].n_iter_.max()<600, 'Solver reached its iteration limit.'

## 4. Check training and validation behaviour
Strong training performance with weaker validation performance supports an overfitting concern. Weak scores on both sets can suggest underfitting, although poor features or labels can also cause it. We inspect the actual gaps before drawing a conclusion. Test scores are not used to change the model.

In [ ]:
behaviour=[]
for split,mask in [('train',train),('validation',validation)]:
    prediction=model.predict(X[mask])
    behaviour.append({'split':split,'accuracy':accuracy_score(y[mask],prediction),
                      'macro_f1':f1_score(y[mask],prediction,average='macro')})
behaviour=pd.DataFrame(behaviour)
behaviour.to_csv(ROOT / 'outputs/binary_behaviour.csv',index=False)
display(behaviour)
fig,ax=plt.subplots(figsize=(6,3.5))
ax.errorbar(cv_results.param_classifier__C.astype(float),cv_results.mean_test_score,
            yerr=cv_results.std_test_score,marker='o',label='CV held-out macro-F1')
ax.plot(cv_results.param_classifier__C.astype(float),cv_results.mean_train_score,marker='o',label='CV training macro-F1')
ax.set(xscale='log',xlabel='C (inverse regularization)',ylabel='Macro-F1',title='Regularization on training folds'); ax.legend()
fig.tight_layout(); fig.savefig(ROOT / 'outputs/binary_regularization.png',dpi=180); plt.show()

## 5. Save and evaluate the selected model
The saved pipeline includes its fitted scaler. The untouched test split supplies the metrics below. We reload the file and verify that its probabilities match.

In [ ]:
model_path=ROOT / 'models/binary.joblib'
joblib.dump(model,model_path,compress=3)
probability=model.predict_proba(X[test])
reloaded=joblib.load(model_path)
np.testing.assert_allclose(reloaded.predict_proba(X[test][:16]),probability[:16],rtol=1e-5,atol=1e-7)
metrics,report=evaluate('binary',y[test],probability,manifest.loc[test,'image_id'].to_numpy(),labels,
                        model_path,elapsed,{'selected_C':search.best_params_['classifier__C']})
display(pd.Series(metrics)); display(report)

## 6. Connect the formula to a real image
The model exposes its learned linear score. We independently calculate the probability and compare it with `predict_proba`.

In [ ]:
score=model.decision_function(X[test][:1])[0]
manual_probability=expit(score)
np.testing.assert_allclose(manual_probability,probability[0,1],rtol=1e-5,atol=1e-7)
print('Logit:',score,'Sigmoid Dog probability:',manual_probability,
      'Prediction at threshold 0.5:',labels[int(manual_probability>=0.5)])
z=np.linspace(-7,7,250)
fig,ax=plt.subplots(figsize=(6,3.5)); ax.plot(z,expit(z)); ax.axhline(0.5,color='gray',linestyle='--')
ax.scatter([score],[manual_probability],color='crimson',label='First test image')
ax.set(xlabel='Linear score z',ylabel='Estimated Dog probability',title='Sigmoid and the 0.5 threshold'); ax.legend()
fig.tight_layout(); fig.savefig(ROOT / 'outputs/sigmoid.png',dpi=180); plt.show()

## 7. Understand the confusion matrix
Dog is the positive class. TN is a Cat predicted as Cat; FP is a Cat predicted as Dog; FN is a Dog predicted as Cat; TP is a Dog predicted as Dog. Precision is TP/(TP+FP), recall is TP/(TP+FN), and F1 is their harmonic mean. Accuracy counts both correct outcomes. We locate one actual image for each available outcome.

In [ ]:
tn,fp,fn,tp=confusion_matrix(y[test],probability.argmax(axis=1),labels=[0,1]).ravel()
print(dict(TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp)))
results=pd.read_csv(ROOT / 'outputs/binary_predictions.csv').merge(manifest[['image_id','path']],on='image_id')
fig,axes=plt.subplots(1,4,figsize=(11,3))
for ax,(title,true,pred) in zip(axes,[('TN',0,0),('FP',0,1),('FN',1,0),('TP',1,1)]):
    candidates=results[(results.truth==true)&(results.prediction==pred)]
    ax.axis('off')
    if len(candidates):
        row=candidates.iloc[0]; ax.imshow(pixels(ROOT / row.path,160))
        ax.set_title(f'{title}: {labels[true]} as {labels[pred]}\nP(Dog)={row.p_Dog:.1%}',fontsize=9)
        print(title,row.image_id,row.path)
fig.tight_layout(); fig.savefig(ROOT / 'outputs/private/binary_outcomes.png',dpi=160); plt.show()

show_predictions('binary',manifest)

## 8. Observations
Training accuracy was 76.15%, validation accuracy 72.91% and test accuracy 72.73%. Test macro-F1 was 0.484. The 74.57% always-Dog baseline has higher accuracy, while this model finds only 8.03% of Cats. Dog recall is 94.79%. These results show why accuracy alone is misleading on an imbalanced dataset.

The training/validation gap is 3.24 percentage points. Weak Cat detection is the main limitation here; the results do not support claiming that the classifier recognizes both animals well. Flattened low-resolution pixels give a limited representation of changes in pose and background.